# Five-day convergence benchmark: CPU SLSQP vs captured exact Hessian

This benchmark runs three arms on the original five-day office calibration problem (2–7 December, 20-minute samples, 20 warm-up steps), allowing each solver to terminate naturally with a high 1000-iteration safety cap:

- **CPU SLSQP**: fast single-shooting with automatic differentiation.
- **CUDA IPOPT/fixed eager**: exact-Hessian collocation using the corrected scaling-and-squaring exponential and compiler-compatible fixed-basis Hessian, without acceleration.
- **CUDA IPOPT/direct graph**: the identical fixed-basis Hessian under direct `torch.cuda.CUDAGraph` replay, using the shared parameter seed.
- **CUDA IPOPT/direct graph, unseeded**: the same fast collocation path starting from the model's default parameter values. Rollout-based boundary-state initialization remains enabled; only the five-iteration SLSQP parameter seed is omitted.

A single 5-iteration CPU SLSQP run creates the shared parameter seed used by the first three arms. Arm timing excludes this common seed, while `total_with_shared_warm_seconds` includes it only where applicable. Quality is scored by a real object-graph rollout using per-sensor RMSE and the pooled weighted objective. The eager and captured seeded CUDA arms are operation-for-operation identical; capture also performs its two-point eager/replay parity check.

Select an A100 GPU runtime and run all cells.

In [ ]:
import json
import os
from pathlib import Path
import platform
import subprocess
import sys

import torch

REPO_URL = "https://github.com/JBjoernskov/Twin4Build.git"
CANDIDATE_REF = "feature/issue-126/reduce-hessian-dispatch"
ROOT = Path("/content/twin4build_cpu_vs_cuda_full")
CHECKOUT = ROOT / "candidate"

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA Colab runtime is required.")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        f"git+{REPO_URL}@{CANDIDATE_REF}",
    ],
    check=True,
)

ROOT.mkdir(parents=True, exist_ok=True)
if CHECKOUT.exists():
    subprocess.run(["git", "fetch", "--quiet", "origin", CANDIDATE_REF], cwd=CHECKOUT, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{CANDIDATE_REF}"], cwd=CHECKOUT, check=True)
else:
    subprocess.run(
        ["git", "clone", "--quiet", "--depth", "1", "--branch", CANDIDATE_REF, REPO_URL, str(CHECKOUT)],
        check=True,
    )

ss_source = (CHECKOUT / "twin4build/systems/utils/discrete_statespace_system.py").read_text()
transcription_source = (CHECKOUT / "twin4build/estimator/_transcription.py").read_text()
if (
    "delta = 2.0 * delta + delta @ delta" not in ss_source
    or "fixed_eager" not in transcription_source
    or "class _CudaGraphCallable" not in transcription_source
):
    raise RuntimeError("Candidate checkout lacks corrected SS or Hessian graph backends.")

props = torch.cuda.get_device_properties(0)
HARDWARE = {
    "cpu": platform.processor() or platform.machine(),
    "cpu_logical": os.cpu_count(),
    "gpu": props.name,
    "gpu_memory_gb": props.total_memory / 1e9,
    "torch": torch.__version__,
    "python": platform.python_version(),
    "candidate_ref": CANDIDATE_REF,
}
print(json.dumps(HARDWARE, indent=2))

In [ ]:
RUNNER = ROOT / "run_full_problem_arm.py"
RUNNER.write_text(r'''
import datetime
import importlib.util
import json
import os
from pathlib import Path
import platform
import subprocess
import sys
import time

import numpy as np
import torch

repo = Path(sys.argv[1]).resolve()
arm = sys.argv[2]
seed_file = Path(sys.argv[3]).resolve()
out_file = Path(sys.argv[4]).resolve()

sys.path.insert(0, str(repo))
os.chdir(repo)

from dateutil import tz
import twin4build as tb
import twin4build.examples as examples_package
import twin4build.examples.utils as example_utils
from twin4build.utils.rgetattr import rgetattr

tb._IS_TESTING = True

STEP = 1200
START = [datetime.datetime(2023, 12, 2, tzinfo=tz.gettz("Europe/Copenhagen"))]
END = [datetime.datetime(2023, 12, 7, tzinfo=tz.gettz("Europe/Copenhagen"))]
N_WARMUP = 20
WARM_ITERS = 5
SLSQP_ITERS = 1000
COLLOCATION_ITERS = 1000
SD = {
    "office_temperature_sensor": 0.05,
    "office_valve_position_sensor": 0.025,
    "office_damper_position_sensor": 0.025,
    "office_co2_sensor": 15.0,
}

example_path = Path(examples_package.__file__).parent / "full_workflow_example.py"
spec = importlib.util.spec_from_file_location("_cpu_cuda_full_workflow", example_path)
workflow = importlib.util.module_from_spec(spec)
spec.loader.exec_module(workflow)


def build_model(device):
    model = tb.Model(id=f"cpu_cuda_full_{arm}")
    model.load(
        semantic_model_filename=example_utils.get_path(
            ["estimator_example", "one_room_example_model.xlsm"]
        ),
        fcn=workflow.fcn,
    )
    model.to(device, torch.float64)
    return model


def build_parameters(model):
    c = model.components
    space, heater = c["office"], c["office_space_heater"]
    hc, cc = c["office_temperature_heating_controller"], c["office_co2_controller"]
    valve = c["office_space_heater_valve"]
    sup, exh = c["office_supply_damper"], c["office_exhaust_damper"]
    occ, wall = c["office_occupancy"], c["office_boundary_wall"]
    det = c["office_occupancy_detector"]
    return [
        (space, "thermal.C_air", 5e5, 1e4, 5e5),
        (space, "thermal.C_wall", 1e6, 1e5, 3e6),
        (wall, "C", 1e6, 1e4, 1e7),
        (space, "thermal.R_out", 0.5, 0.01, 1),
        (space, "thermal.R_in", 0.1, 0.01, 1),
        (wall, "R_a", 0.04, 1e-4, 1),
        (wall, "R_b", 0.04, 1e-4, 1),
        (space, "thermal.f_wall", 0.1, 0, 10),
        (space, "thermal.f_air", 0.1, 0, 10),
        (space, "thermal.Q_occ_gain", 100.0, 10, 200),
        (heater, "thermalMassHeatCapacity", 1e4, 1e3, 2e5),
        (heater, "UA", None, 1, 100),
        (hc, "kp", 0.005, 1e-5, 1, "private"),
        (cc, "kp", 0.0001, 1e-5, 1, "private"),
        ([hc, cc], "Ti", 30, 1, 300, "private"),
        ([hc, cc], "Td", 0, 0, 1, "private"),
        (valve, "waterFlowRateMax", 0.001, 1e-6, 0.1),
        (valve, "valveAuthority", 1, 0.4, 1),
        ([sup, occ.supply_damper], "a", 1, 1, 10, "shared"),
        ([sup, occ.supply_damper], "nominalAirFlowRate", 0.1, 1e-5, 1, "shared"),
        ([exh, occ.exhaust_damper], "a", 1, 1, 10, "shared"),
        ([exh, occ.exhaust_damper], "nominalAirFlowRate", 0.1, 1e-5, 1, "shared"),
        ([space, occ], "mass.V", 65, 50, 80, "shared"),
        ([space, occ], "mass.G_occ", 1e-6, 1e-6, 1e-5, "shared"),
        ([space, occ], "mass.m_inf", 0.001, 1e-4, 0.01, "shared"),
        (det, "threshold", 1.0, 0.02, 5.0),
    ]


def build_measurements(model):
    c = model.components
    return [
        (c["office_valve_position_sensor"], 0.025),
        (c["office_temperature_sensor"], 0.05),
        (c["office_damper_position_sensor"], 0.025),
        (c["office_co2_sensor"], 15.0),
    ]


def read_parameter_values(entries):
    values = []
    for comps, attr, _x0, lo, hi, *_ in entries:
        comp = comps[0] if isinstance(comps, list) else comps
        value = float(rgetattr(comp, attr).get().reshape(-1)[0])
        eps = 1e-9 * (hi - lo)
        values.append(min(max(value, lo + eps), hi - eps))
    return values


def apply_seed(entries, values):
    if len(entries) != len(values):
        raise RuntimeError("Shared seed does not match the parameter specification")
    return [
        (entry[0], entry[1], value, entry[3], entry[4], *entry[5:])
        for entry, value in zip(entries, values)
    ]


def score_rollout(model, simulator, initial_state):
    def seed_states():
        for component_id, state in initial_state.items():
            model.get_component(component_id).set_state(state)

    model.set_save_simulation_result(flag=True)
    started = time.perf_counter()
    simulator.simulate(
        step_size=STEP,
        start_time=START,
        end_time=END,
        show_progress_bar=False,
        after_initialize=seed_states if initial_state else None,
    )
    rollout_seconds = time.perf_counter() - started

    rmse = {}
    pooled = 0.0
    for component_id, sd in SD.items():
        component = model.components[component_id]
        simulated = component.output["measuredValue"].history()[:, 0, 0].detach().cpu().numpy()[N_WARMUP:]
        actual = component.time_series_input.values[:, 0, 0].detach().cpu().numpy()[N_WARMUP:]
        value = float(np.sqrt(np.mean((simulated - actual) ** 2)))
        short_name = component_id.replace("office_", "").replace("_sensor", "")
        rmse[short_name] = value
        pooled += (value / sd) ** 2
    return rmse, pooled, rollout_seconds


if arm == "seed":
    model = build_model("cpu")
    estimator = tb.Estimator(tb.Simulator(model))
    parameters = build_parameters(model)
    started = time.perf_counter()
    result = estimator.estimate(
        START, END, STEP, parameters, build_measurements(model),
        n_warmup=N_WARMUP,
        method=("scipy", "SLSQP", "ad"),
        options={"maxiter": WARM_ITERS, "fast": True},
    )
    seed_seconds = time.perf_counter() - started
    payload = {
        "ref": subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"], cwd=repo, text=True
        ).strip(),
        "values": read_parameter_values(parameters),
        "seconds": seed_seconds,
        "iterations": None if result.get("iterations") is None else int(result["iterations"]),
        "objective": None if result.get("final_objective") is None else float(result["final_objective"]),
        "success": None if result.get("success") is None else bool(result["success"]),
        "message": str(result.get("message")),
    }
    seed_file.write_text(json.dumps(payload, indent=2))
    print(json.dumps(payload, indent=2))
    raise SystemExit(0)

seed = json.loads(seed_file.read_text())
device = "cpu" if arm == "cpu_slsqp" else "cuda"
cuda_backends = {
    "cuda_fixed_eager": "fixed_eager",
    "cuda_direct_graph": "cuda_graph",
    "cuda_direct_graph_unseeded": "cuda_graph",
}
if arm != "cpu_slsqp" and arm not in cuda_backends:
    raise ValueError(f"Unknown arm {arm!r}")
if arm in cuda_backends and os.environ.get("TWIN4BUILD_TRANSFORM_MATRIX_EXP") != "ss":
    raise RuntimeError(f"{arm} requires the corrected SS matrix exponential")

model = build_model(device)
simulator = tb.Simulator(model)
estimator = tb.Estimator(simulator)
parameters = build_parameters(model)
uses_shared_seed = arm != "cuda_direct_graph_unseeded"
if uses_shared_seed:
    parameters = apply_seed(parameters, seed["values"])
measurements = build_measurements(model)

if device == "cuda":
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
started = time.perf_counter()
if arm == "cpu_slsqp":
    result = estimator.estimate(
        START, END, STEP, parameters, measurements,
        n_warmup=N_WARMUP,
        method=("scipy", "SLSQP", "ad"),
        options={"maxiter": SLSQP_ITERS, "fast": True},
    )
else:
    result = estimator.estimate(
        START, END, STEP, parameters, measurements,
        n_warmup=N_WARMUP,
        method=("casadi", "ipopt", "ad", "collocation"),
        options={
            "maxiter": COLLOCATION_ITERS,
            "exact_hessian": True,
            "compile_hessian": True,
            "compile_hessian_backend": cuda_backends[arm],
            "early_stopping": False,
            "boundary_state_init": "rollout",
        },
    )
if device == "cuda":
    torch.cuda.synchronize()
estimate_seconds = time.perf_counter() - started

audit = result.get("transcription_audit", {}) or {}
initial_state = result.get("estimated_initial_state", {}) or {}
rmse, pooled, rollout_seconds = score_rollout(model, simulator, initial_state)

row = {
    "arm": arm,
    "ref": subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=repo, text=True).strip(),
    "device": device,
    "matrix_exp": "ss" if device == "cuda" else None,
    "hessian_backend": cuda_backends.get(arm),
    "hardware": torch.cuda.get_device_name(0) if device == "cuda" else (platform.processor() or platform.machine()),
    "hours": 120,
    "steps": int((END[0] - START[0]).total_seconds() / STEP),
    "warmup_steps": N_WARMUP,
    "iteration_limit": SLSQP_ITERS if arm == "cpu_slsqp" else COLLOCATION_ITERS,
    "iterations": None if result.get("iterations") is None else int(result["iterations"]),
    "success": None if result.get("success") is None else bool(result["success"]),
    "message": str(result.get("message")),
    "final_objective": None if result.get("final_objective") is None else float(result["final_objective"]),
    "parameter_seed": "cpu_slsqp_5" if uses_shared_seed else "model_defaults",
    "estimate_seconds": estimate_seconds,
    "shared_warm_seconds": seed["seconds"] if uses_shared_seed else 0.0,
    "total_with_shared_warm_seconds": estimate_seconds + (seed["seconds"] if uses_shared_seed else 0.0),
    "rollout_audit_seconds": rollout_seconds,
    "pooled_rollout_objective": pooled,
    "max_defect": None if audit.get("max_defect") is None else float(audit["max_defect"]),
    "peak_cuda_memory_gb": torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else None,
    **{f"rmse_{name}": value for name, value in rmse.items()},
}
out_file.write_text(json.dumps(row, indent=2))
print(json.dumps(row, indent=2))
''')
print(f"wrote {RUNNER}")

In [ ]:
SEED_FILE = ROOT / "shared_cpu_seed.json"
RESULT_FILES = {
    "cpu_slsqp": ROOT / "cpu_slsqp_convergence_1000.json",
    "cuda_fixed_eager": ROOT / "cuda_fixed_eager_convergence_1000.json",
    "cuda_direct_graph": ROOT / "cuda_direct_graph_convergence_1000.json",
    "cuda_direct_graph_unseeded": ROOT / "cuda_direct_graph_unseeded_convergence_1000.json",
}
CURRENT_REF = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], cwd=CHECKOUT, text=True
).strip()
SEED_REFS = {"35ec36c", "0c02087", CURRENT_REF}


def seed_is_current():
    if not SEED_FILE.exists():
        return False
    try:
        return json.loads(SEED_FILE.read_text()).get("ref") in SEED_REFS
    except (json.JSONDecodeError, OSError):
        return False


def result_is_current(path):
    if not path.exists():
        return False
    try:
        row = json.loads(path.read_text())
        return (
            row.get("ref") in {"e8caeef", CURRENT_REF}
            and row.get("iteration_limit") == 1000
        )
    except (json.JSONDecodeError, OSError):
        return False


def run_process(arm, out_file):
    env = os.environ.copy()
    env["PYTHONPATH"] = str(CHECKOUT) + os.pathsep + env.get("PYTHONPATH", "")
    if arm.startswith("cuda_"):
        env["TWIN4BUILD_TRANSFORM_MATRIX_EXP"] = "ss"
    else:
        env.pop("TWIN4BUILD_TRANSFORM_MATRIX_EXP", None)
    completed = subprocess.run(
        [sys.executable, str(RUNNER), str(CHECKOUT), arm, str(SEED_FILE), str(out_file)],
        cwd=CHECKOUT,
        env=env,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout, flush=True)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr, flush=True)
    if completed.returncode:
        raise RuntimeError(
            f"{arm} failed with exit code {completed.returncode}.\n"
            + "\n".join(completed.stderr.splitlines()[-80:])
        )


if not seed_is_current():
    print("Running shared 5-iteration CPU SLSQP seed ...", flush=True)
    run_process("seed", ROOT / "unused_seed_result.json")
else:
    print("Reusing current shared CPU seed.")

for arm, result_file in RESULT_FILES.items():
    if result_is_current(result_file):
        print(f"Reusing current {arm} result.")
        continue
    print(f"\nRunning {arm} to convergence on the full five-day problem ...", flush=True)
    run_process(arm, result_file)

rows = [json.loads(path.read_text()) for path in RESULT_FILES.values()]
for row in rows:
    row.setdefault(
        "parameter_seed",
        "model_defaults" if row["arm"] == "cuda_direct_graph_unseeded" else "cpu_slsqp_5",
    )
print("\nAll convergence arms completed.")

In [ ]:
import pandas as pd

summary = pd.DataFrame(rows)
cpu = summary.loc[summary.arm == "cpu_slsqp"].iloc[0]
eager = summary.loc[summary.arm == "cuda_fixed_eager"].iloc[0]
graphed = summary.loc[summary.arm == "cuda_direct_graph"].iloc[0]
unseeded = summary.loc[summary.arm == "cuda_direct_graph_unseeded"].iloc[0]
summary["estimate_speedup_vs_cpu_slsqp"] = cpu.estimate_seconds / summary.estimate_seconds
summary["total_speedup_vs_cpu_slsqp"] = (
    cpu.total_with_shared_warm_seconds / summary.total_with_shared_warm_seconds
)

columns = [
    "arm", "parameter_seed", "matrix_exp", "hessian_backend", "ref", "hardware",
    "iteration_limit", "iterations", "success", "message", "final_objective", "estimate_seconds",
    "shared_warm_seconds", "total_with_shared_warm_seconds",
    "estimate_speedup_vs_cpu_slsqp", "pooled_rollout_objective", "max_defect",
    "rmse_temperature", "rmse_valve_position", "rmse_damper_position", "rmse_co2",
    "peak_cuda_memory_gb",
]
pd.set_option("display.max_columns", None)
display(summary[columns])

for label, arm in (
    ("fixed eager, seeded", eager),
    ("direct graph, seeded", graphed),
    ("direct graph, unseeded", unseeded),
):
    print(
        f"CUDA exact collocation ({label}) / CPU SLSQP:\n"
        f"  estimation speedup: {cpu.estimate_seconds / arm.estimate_seconds:.3f}x\n"
        f"  pooled rollout objective: {arm.pooled_rollout_objective:.6g}\n"
        f"  maximum continuity defect: {arm.max_defect:.3e}\n"
        f"  iterations: {int(arm.iterations)}"
    )

print(
    f"Direct graph / fixed-eager estimation speedup: "
    f"{eager.estimate_seconds / graphed.estimate_seconds:.3f}x\n"
    f"Iteration-count difference: {int(graphed.iterations - eager.iterations):+d}\n"
    f"Rollout-objective difference: "
    f"{graphed.pooled_rollout_objective - eager.pooled_rollout_objective:+.6g}\n"
    f"Maximum-defect difference: {graphed.max_defect - eager.max_defect:+.3e}"
)
print(
    "\nEffect of omitting the five-iteration parameter seed:\n"
    f"  time ratio (unseeded/seeded): {unseeded.estimate_seconds / graphed.estimate_seconds:.3f}x\n"
    f"  iteration difference: {int(unseeded.iterations - graphed.iterations):+d}\n"
    f"  rollout-objective difference: "
    f"{unseeded.pooled_rollout_objective - graphed.pooled_rollout_objective:+.6g}\n"
    f"  maximum-defect difference: {unseeded.max_defect - graphed.max_defect:+.3e}"
)

all_arms = (
    ("CPU SLSQP", cpu),
    ("CUDA fixed eager", eager),
    ("CUDA direct graph", graphed),
    ("CUDA direct graph unseeded", unseeded),
)
for label, arm in all_arms:
    if not bool(arm.success):
        print(f"WARNING: {label} hit the 1000-iteration safety cap or otherwise failed.")
for label, arm in all_arms[1:]:
    if pd.isna(arm.max_defect) or arm.max_defect > 1e-6:
        print(f"WARNING: {label} is not rollout-feasible at 1e-6; reject final quality comparison.")